In [50]:
import pandas as pd
import requests

url = 'https://data.nationalbank.kz/api/report/stat-data-locale'

headers = {
    'Accept': 'application/json'
}
params = {
    'form_id': '299',
    'date_from': '2023-01-01',
    'date_to': '2023-01-10',
    'lang': 'en'
}

resp = requests.get(url, headers = headers, params = params)
data = resp.json()
df = pd.json_normalize(data)

#####normlising attributions 
def attrs_to_dict(attrs):
    if not isinstance(attrs, list):
        return {}
    return {d.get("attrCode"): d.get("valueCode") for d in attrs}

attrs_df = df["attributes"].apply(attrs_to_dict).apply(pd.Series)

df = pd.concat([df.drop(columns=["attributes"]), attrs_df], axis=1)
#######

#drop unnnecessery columns and change column name and add additional column 
df = df.drop(columns = ['type', 'category'])
df = df.rename(columns={'fx_currency_pair': 'currency', 'reportDate': 'report_date'})
df["updated_at"] = pd.Timestamp.now()

#to csv:
df.to_csv('exchange_rate.csv')

